In [18]:
import pandas as pd
import json
from geopy.distance import geodesic
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


train_values = pd.read_csv("train_values.csv")
train_labels = pd.read_csv("train_labels.csv")


df_raw = pd.merge(train_values, train_labels, on="building_id")


non_structural_cols = [
    'building_id', 
    'geo_level_1_id',  
    'geo_level_3_id', 
    'legal_ownership_status',
    'position',
    'count_families'
    ]

secondary_use_cols = [col for col in df_raw.columns if col.startswith('has_secondary_use')]
cols_to_drop = non_structural_cols + secondary_use_cols
df_final = df_raw.drop(columns= cols_to_drop)


print(f"Shape after filtering: {df_final.shape}")

Shape after filtering: (260601, 23)


In [19]:
EPICENTER = (28.231, 84.731)


with open("GEO_LEVEL_2_MAP.json", 'r') as f:
    geo_dict = json.load(f)


distance_map = {}
for k, v in geo_dict.items():
    if isinstance(v, dict) and 'lat' in v and 'lon' in v:
        point = (v['lat'], v['lon'])
        # Calculate distance and store it with an integer key
        distance_map[int(k)] = np.round(geodesic(point, EPICENTER).kilometers, 2)
    else:
        distance_map[int(k)] = None


df_final["epicenter_distance"] = df_final["geo_level_2_id"].map(distance_map)
df_final = df_final.drop(columns=['geo_level_2_id'])



# 1. Model how shaking force decays over distance
df_final['hazard_force'] = 1.0 - (np.log1p(df_final['epicenter_distance']) / 5.0)
df_final['hazard_force'] = np.clip(df_final['hazard_force'], 0.05, 1.0)

# 2. Normalize the categorical damage grade (1, 2, 3) to a 0.0 - 1.0 range
df_final['damage_norm'] = (df_final['damage_grade'] - 1) / 2.0

# 3. Calculate vulnerability (Damage sustained relative to the force felt)
df_final['vulnerability'] = df_final['damage_norm'] / (df_final['hazard_force'] + 0.1)

# 4. Invert vulnerability to get Resilience, and scale from 0 to 100
raw_resilience = 1.0 / (df_final['vulnerability'] + 0.2)

min_r = raw_resilience.min()
max_r = raw_resilience.max()

df_final['resilience_score'] = ((raw_resilience - min_r) / (max_r - min_r)) * 100.0

epicenter_series = df_final['epicenter_distance']

df_final = df_final.drop(columns=["damage_grade", 'hazard_force', 'damage_norm', 'vulnerability', 'epicenter_distance'])
df_final.to_csv('resilence_dataset.csv')
df_final

,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,plan_configuration,...,has_superstructure_stone_flag,has_superstructure_cement_mortar_stone,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,resilience_score
0,2,30,6,5,t,r,n,f,q,d,...,0,0,0,0,0,0,0,0,0,0.000000
1,2,10,8,7,o,r,n,x,q,d,...,0,0,0,0,0,0,0,0,0,3.032680
2,2,10,5,5,t,r,n,f,x,d,...,0,0,0,0,0,0,0,0,0,0.000000
3,2,10,6,5,t,r,n,f,x,d,...,0,0,0,0,1,1,0,0,0,13.079632
4,3,30,8,9,t,r,n,f,x,d,...,0,0,0,0,0,0,0,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260596,1,55,6,3,n,r,n,f,j,q,...,0,0,0,0,0,0,0,0,0,10.094963
260597,2,0,6,5,t,r,n,f,q,d,...,0,0,0,0,0,0,0,0,0,1.272464
260598,3,55,6,7,t,r,q,f,q,d,...,0,0,0,0,0,0,0,0,0,0.537086
260599,2,10,14,6,t,r,x,v,s,d,...,0,0,0,1,0,0,0,0,0,4.688472


In [20]:
for i in range(0,100,5):
    print(
        f"{i}% to {i+5}%: {df_final['resilience_score'][(df_final['resilience_score']>i)\
                                         & (df_final['resilience_score']<=i+5)].count()} rows"
        )

0% to 5%: 136598 rows
5% to 10%: 45428 rows
10% to 15%: 11476 rows
15% to 20%: 1712 rows
20% to 25%: 450 rows
25% to 30%: 0 rows
30% to 35%: 0 rows
35% to 40%: 0 rows
40% to 45%: 0 rows
45% to 50%: 0 rows
50% to 55%: 0 rows
55% to 60%: 0 rows
60% to 65%: 0 rows
65% to 70%: 0 rows
70% to 75%: 0 rows
75% to 80%: 0 rows
80% to 85%: 0 rows
85% to 90%: 0 rows
90% to 95%: 0 rows
95% to 100%: 25124 rows


In [25]:
epicenter_series.corr(df_final['resilience_score'], method= 'spearman')

np.float64(-0.5647451452182911)

TRAINING

In [27]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split


# Load your preprocessed dataframe containing distance calculations
df = pd.read_csv('resilence_dataset.csv')

X = df_final.drop(columns= ['resilience_score'])
y = df_final['resilience_score']
print("Features the model will train on:", list(X.columns))

Features the model will train on: ['count_floors_pre_eq', 'age', 'area_percentage', 'height_percentage', 'land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'plan_configuration', 'has_superstructure_adobe_mud', 'has_superstructure_mud_mortar_stone', 'has_superstructure_stone_flag', 'has_superstructure_cement_mortar_stone', 'has_superstructure_mud_mortar_brick', 'has_superstructure_cement_mortar_brick', 'has_superstructure_timber', 'has_superstructure_bamboo', 'has_superstructure_rc_non_engineered', 'has_superstructure_rc_engineered', 'has_superstructure_other']


In [ ]:
# Split into train and evaluation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# Instantiate and fit a high-performance tree regressor
regressor = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.05,
    n_estimators=300,
    objective='reg:squarederror',
    random_state=42
)

regressor.fit(X_train, y_train)
print("Model training complete. Your resilience engine is ready for global deployment!")